<a href="https://colab.research.google.com/github/listenmusiceveryday/AIB6_Progress_Note/blob/main/tongue_medgemma_zeroshot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👅 Tongue Health — MedGemma Zero-Shot Baseline

**Goal:** ส่ง Test Set ให้ MedGemma-4b-it predict แบบ zero-shot แล้วคำนวณ F1/Precision/Recall เทียบกับ ground truth

**Hardware:** T4 (15 GB VRAM) — Google Colab Free Tier

> ⚠️ ต้องขอ access MedGemma บน Hugging Face ก่อน: https://huggingface.co/google/medgemma-4b-it

## 0️⃣ Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 1️⃣ Install Packages

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q datasets scikit-learn Pillow tqdm pandas
!pip install -q iterative-stratification
print("✓ Done")

## 2️⃣ Hugging Face Login
> ต้องใช้ token ที่มีสิทธิ์ access `google/medgemma-4b-it`

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3️⃣ Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    # ── Data ──────────────────────────────────────────────────────────────
    DATA_DIR:  str = "/content/drive/MyDrive/kaggle_dataset/shezhenv3-txt/images"
    CSV_FILE:  str = "/content/drive/MyDrive/kaggle_dataset/shezhenv3-txt/tongue_real.csv"
    IMAGE_COL: str = "Image_Name"
    LABEL_COLS: List[str] = field(default_factory=lambda: [
        'Color_Pink', 'Color_Red',
        'Coating_None', 'Coating_White', 'Coating_Yellow',
        'Texture_None', 'Texture_Geographic', 'Texture_Cracked', 'Texture_Dentate'
    ])

    # ── Split ─────────────────────────────────────────────────────────────
    TEST_RATIO:  float = 0.10
    RANDOM_SEED: int   = 42

    # ── Model ─────────────────────────────────────────────────────────────
    MODEL_ID: str = "google/medgemma-4b-it"

    # ── Inference ─────────────────────────────────────────────────────────
    MAX_NEW_TOKENS: int   = 256
    BATCH_SIZE:     int   = 1

    # ── Output ────────────────────────────────────────────────────────────
    OUTPUT_DIR: str = "/content/medgemma_zeroshot_output"

CFG = Config()
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print(f"✓ Labels ({len(CFG.LABEL_COLS)}): {CFG.LABEL_COLS}")
print(f"✓ Model : {CFG.MODEL_ID}")
print(f"✓ Output: {CFG.OUTPUT_DIR}")

## 4️⃣ Load Data & Split Test Set
> ใช้ Multilabel Stratified Split เพื่อให้ test set มี label distribution ตรงกับ dataset รวม

In [ ]:
import pandas as pd
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

df = pd.read_csv(CFG.CSV_FILE)
print(f"Total samples : {len(df)}")
display(df.head(3))

print("\nLabel distribution:")
for col in CFG.LABEL_COLS:
    n = df[col].sum()
    print(f"  {col:25s}: {n:4d} ({n/len(df)*100:.1f}%)")

In [ ]:
X = df[[CFG.IMAGE_COL]].values
y = df[CFG.LABEL_COLS].values

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=CFG.TEST_RATIO, random_state=CFG.RANDOM_SEED
)
train_val_idx, test_idx = next(msss.split(X, y))
test_df = df.iloc[test_idx].reset_index(drop=True)

print(f"Test set : {len(test_df)} samples ({len(test_df)/len(df)*100:.1f}%)")
print("\nTest label distribution:")
for col in CFG.LABEL_COLS:
    n = test_df[col].sum()
    print(f"  {col:25s}: {n:3d} ({n/len(test_df)*100:.1f}%)")

## 5️⃣ Load MedGemma-4b-it
> ใช้ `bfloat16` (native dtype ของ Gemma3) — ใช้ VRAM ประมาณ 8-9 GB บน T4

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

print(f"Loading {CFG.MODEL_ID} ...")

processor = AutoProcessor.from_pretrained(CFG.MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    CFG.MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16,   # ✅ native dtype ของ Gemma3 — ทำงานได้ถูกต้องบน T4
)
model.eval()

print(f"\n✓ Model loaded")
print(f"  Device : {next(model.parameters()).device}")
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM   : {used:.1f} / {total:.1f} GB")

## 6️⃣ Prompt
> ออกแบบให้ model ตอบ JSON เท่านั้น เพื่อให้ parse ง่าย

In [ ]:
def build_prompt(label_cols):
    groups = {}
    for col in label_cols:
        prefix, val = col.split("_", 1)
        groups.setdefault(prefix, []).append(val)

    group_lines = "\n".join(
        f"- {grp}: options are {vals} (1 = present, 0 = absent)"
        for grp, vals in groups.items()
    )
    json_keys = "{" + ", ".join(f'"{c}": 0 or 1' for c in label_cols) + "}"

    return f"""You are a medical AI assistant specializing in tongue diagnosis.
Examine the tongue image carefully and classify each feature below.

Features to classify:
{group_lines}

Rules:
- Color: EXACTLY ONE of Color_Pink or Color_Red must be 1
- Coating: EXACTLY ONE of Coating_None, Coating_White, Coating_Yellow must be 1
- Texture: EXACTLY ONE of Texture_None, Texture_Geographic, Texture_Cracked, Texture_Dentate must be 1
- All other values must be 0

Respond ONLY with a valid JSON object. No explanation, no markdown, no extra text.
Format: {json_keys}"""

PROMPT = build_prompt(CFG.LABEL_COLS)
print("=== PROMPT ===")
print(PROMPT)

## 7️⃣ Inference Functions

In [ ]:
import json, re
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

def parse_json_response(text: str, label_cols: list) -> dict:
    """Extract JSON from model response. Returns None if parse fails."""
    if not text:
        return None
    try:
        # กรณีมี markdown code block ```json ... ```
        match = re.search(r'```(?:json)?\s*({.*?})\s*```', text, re.DOTALL)
        if match:
            return json.loads(match.group(1))
        # กรณี JSON ล้วน — เอาอันสุดท้ายใน text
        matches = list(re.finditer(r'{[^{}]+}', text, re.DOTALL))
        if matches:
            return json.loads(matches[-1].group(0))
    except Exception:
        pass
    return None

def response_to_labels(parsed: dict, label_cols: list) -> list:
    """Convert parsed dict → list[int]. Returns -1 for missing/unparseable values."""
    if parsed is None:
        return [-1] * len(label_cols)
    result = []
    for col in label_cols:
        val = parsed.get(col, -1)
        try:
            result.append(int(val))
        except (ValueError, TypeError):
            result.append(-1)
    return result

@torch.no_grad()
def predict_single(image_path, prompt, model, processor, cfg):
    """Run zero-shot inference on a single image. Returns (response_text, error)."""
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return None, f"Image load error: {e}"

    try:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text": prompt},
                ],
            }
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)   # ✅ ไม่แปลง dtype — ให้ model จัดการเอง

        input_len = inputs["input_ids"].shape[-1]

        outputs = model.generate(
            **inputs,
            max_new_tokens=cfg.MAX_NEW_TOKENS,
            do_sample=False,
        )

        response = processor.decode(
            outputs[0][input_len:],
            skip_special_tokens=True
        )
        return response, None

    except Exception as e:
        return None, str(e)

print("✓ Functions ready")

## 8️⃣ Run Inference
> ประมาณ 45-60 นาที สำหรับ ~320 รูปบน T4

In [ ]:
results = []
errors  = []
data_dir = Path(CFG.DATA_DIR)

print(f"Running zero-shot on {len(test_df)} images ...")

for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Predicting"):
    img_path = data_dir / row[CFG.IMAGE_COL]
    raw_text, error = predict_single(img_path, PROMPT, model, processor, CFG)

    if error:
        errors.append({"idx": i, "image": row[CFG.IMAGE_COL], "error": error})
        pred_labels = [-1] * len(CFG.LABEL_COLS)
        raw_text = ""
    else:
        parsed     = parse_json_response(raw_text, CFG.LABEL_COLS)
        pred_labels = response_to_labels(parsed, CFG.LABEL_COLS)

    results.append({
        "image":      row[CFG.IMAGE_COL],
        "raw_output": raw_text,
        "pred":       pred_labels,
        "true":       [int(row[c]) for c in CFG.LABEL_COLS],
    })

print(f"\n✓ Done — {len(results)} processed, {len(errors)} errors")
if errors:
    print("\n⚠️ Errors (first 5):")
    for e in errors[:5]:
        print(f"   {e['image']}: {e['error']}")

## 9️⃣ Compute Metrics

In [ ]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    hamming_loss, classification_report
)

valid_preds, valid_trues, skipped = [], [], 0
for r in results:
    if -1 in r["pred"]:
        skipped += 1
        continue
    valid_preds.append(r["pred"])
    valid_trues.append(r["true"])

y_pred = np.array(valid_preds)
y_true = np.array(valid_trues)

print(f"Valid : {len(y_pred)} / {len(results)}  |  Skipped: {skipped}")

if len(y_pred) == 0:
    print("\n❌ ไม่มี valid predictions — ตรวจสอบ sample outputs ด้านล่าง")
else:
    metrics = {
        "f1_micro"    : f1_score(y_true, y_pred, average="micro",   zero_division=0),
        "f1_macro"    : f1_score(y_true, y_pred, average="macro",   zero_division=0),
        "f1_samples"  : f1_score(y_true, y_pred, average="samples", zero_division=0),
        "precision"   : precision_score(y_true, y_pred, average="micro", zero_division=0),
        "recall"      : recall_score(y_true, y_pred,    average="micro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "exact_match" : float((y_pred == y_true).all(axis=1).mean()),
        "n_valid"     : len(y_pred),
        "n_total"     : len(results),
        "n_skipped"   : skipped,
    }

    print("=" * 55)
    print("  ZERO-SHOT RESULTS  (MedGemma-4b-it)")
    print("=" * 55)
    print(f"  F1 Micro      : {metrics['f1_micro']:.4f}")
    print(f"  F1 Macro      : {metrics['f1_macro']:.4f}")
    print(f"  F1 Samples    : {metrics['f1_samples']:.4f}")
    print(f"  Precision     : {metrics['precision']:.4f}")
    print(f"  Recall        : {metrics['recall']:.4f}")
    print(f"  Hamming Loss  : {metrics['hamming_loss']:.4f}")
    print(f"  Exact Match   : {metrics['exact_match']:.4f}")
    print("=" * 55)

    print("\nPer-Label F1:")
    print("-" * 45)
    per_label_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    for label, score in zip(CFG.LABEL_COLS, per_label_f1):
        bar = "█" * int(score * 20)
        print(f"  {label:25s}: {score:.4f}  {bar}")

    print()
    print(classification_report(y_true, y_pred, target_names=CFG.LABEL_COLS, zero_division=0))

## 🔟 Save Results

In [ ]:
import json

# Raw results JSON
results_path = f"{CFG.OUTPUT_DIR}/zeroshot_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

# Metrics JSON
metrics_path = f"{CFG.OUTPUT_DIR}/zeroshot_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

# Predictions CSV (ใช้เปรียบกับ fine-tuned model ทีหลัง)
pred_cols = {f"pred_{c}": [r["pred"][i] for r in results] for i, c in enumerate(CFG.LABEL_COLS)}
true_cols = {f"true_{c}": [r["true"][i] for r in results] for i, c in enumerate(CFG.LABEL_COLS)}
pred_df = pd.DataFrame({"image": [r["image"] for r in results], **pred_cols, **true_cols})
csv_path = f"{CFG.OUTPUT_DIR}/zeroshot_predictions.csv"
pred_df.to_csv(csv_path, index=False)

print(f"✓ Saved to {CFG.OUTPUT_DIR}/")
print(f"  - zeroshot_results.json")
print(f"  - zeroshot_metrics.json")
print(f"  - zeroshot_predictions.csv")

## 🔍 Inspect Sample Outputs
> ดู raw output เพื่อ debug ถ้า parse rate ต่ำ

In [ ]:
print("Sample outputs (first 5):")
print("=" * 70)
for r in results[:5]:
    print(f"Image : {r['image']}")
    print(f"True  : {dict(zip(CFG.LABEL_COLS, r['true']))}")
    print(f"Pred  : {dict(zip(CFG.LABEL_COLS, r['pred']))}")
    print(f"Raw   : {r['raw_output'][:200]}")
    print("-" * 70)